In [3]:
#!/usr/bin/env python3
"""
Calibrate zero current measurements using electrode_8mA files as baseline.
Calculate baseline current for each rfd/tbs combination and adjust all traces in electrode_local.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
import json

# ===== CONFIGURATION =====
CALIBRATION_FOLDER = Path("/Users/xylu/Desktop/Data/electrode_8mA")
DATA_FOLDER = Path("/Users/xylu/Desktop/Data/electrode_local")
OUTPUT_FOLDER = Path("/Users/xylu/Desktop/Data/electrode_local_calibrated")
BASELINE_FILE = Path("/Users/xylu/Desktop/Data/baseline_currents.json")

# Smoothing parameters (same as in photocurrent processing)
smoothing_window_size = 300
baseline_time_min = -1200  # µs
baseline_time_max = -200   # µs

# Time shifts for file types
time_shifts = {
    'rfd4tbs2': 8.707,
    'rfd10tbs1': 10.320,
    'rfd11tbs1': -0.205,
    'rfd11tbs2': -0.205
}

# ===== HELPER FUNCTIONS =====

def extract_file_id(filename):
    """Extract rfd/tbs identifier from filename.
    E.g., '20260131_220406_rfd11tbs1.csv' -> 'rfd11tbs1'
    """
    stem = filename.stem
    parts = stem.split('_')
    if len(parts) >= 3:
        return parts[-1]  # rfd*tbs*
    return None


def read_and_process_calibration_file(file_path):
    """Read calibration file and extract baseline current.
    
    Returns:
        dict: Contains 'baseline_current', 'channels', 'raw_trace', 'smoothed_trace'
    """
    filename = file_path.name
    file_id = extract_file_id(file_path)
    is_rfd11 = 'rfd11' in filename
    
    # Detect header row
    skip_rows = 4
    ntrigger_val = None
    try:
        with open(file_path, 'r', errors='ignore') as f:
            for i in range(15):
                line = f.readline()
                if "Ntrigger" in line:
                    parts = line.split(',')
                    if len(parts) >= 2:
                        ntrigger_val = parts[1].strip()
                if line.startswith('CH1'):
                    skip_rows = i
                    break
    except Exception as e:
        print(f"Warning: Could not read header from {filename}: {e}")
        pass
    
    # Read channels
    try:
        if is_rfd11:
            df = pd.read_csv(file_path, skiprows=skip_rows, usecols=['CH1', 'CH2', 'CH3'])
            channels = ['CH1', 'CH2', 'CH3']
        else:
            df = pd.read_csv(file_path, skiprows=skip_rows, usecols=['CH1'])
            channels = ['CH1']
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None
    
    # Calculate time in microseconds
    time_us = np.arange(len(df)) * 75e-9 * 1e6 - 1400
    
    # Apply time shift
    if file_id in time_shifts:
        time_us = time_us + time_shifts[file_id]
    
    # Process each channel to calculate baseline
    baseline_currents = {}
    
    for ch_name in channels:
        try:
            ch_data = df[ch_name].values.copy()
            
            # Convert raw data to current in mA
            current_mA = ch_data / 10 / 50 * 1000  # 10x probe, 50Ω resistor, to mA
            
            # Apply smoothing filter
            smoothed_current = np.convolve(current_mA, np.ones(smoothing_window_size)/smoothing_window_size, mode='same')
            
            # Find baseline region (-1200 to -200 µs)
            baseline_mask = (time_us >= baseline_time_min) & (time_us <= baseline_time_max)
            baseline_region = smoothed_current[baseline_mask]
            
            if len(baseline_region) > 0:
                baseline_current = np.mean(baseline_region)
                baseline_currents[ch_name] = float(baseline_current)
            else:
                print(f"Warning: No data in baseline region for {filename} {ch_name}")
                baseline_currents[ch_name] = 0.0
                
        except Exception as e:
            print(f"Error processing {filename} {ch_name}: {e}")
            baseline_currents[ch_name] = 0.0
    
    return {
        'file_id': file_id,
        'filename': filename,
        'channels': channels,
        'baseline_currents': baseline_currents,
        'is_rfd11': is_rfd11
    }


def calculate_baseline_map():
    """Process all calibration files and create a baseline map.
    Groups by file_id (rfd*tbs*) and averages multiple files.
    
    Returns:
        dict: Map of file_id -> baseline currents for each channel
    """
    baseline_map = defaultdict(lambda: defaultdict(list))
    
    if not CALIBRATION_FOLDER.exists():
        print(f"Error: Calibration folder not found: {CALIBRATION_FOLDER}")
        return {}
    
    cal_files = sorted(CALIBRATION_FOLDER.glob("*.csv"))
    print(f"\nReading {len(cal_files)} calibration files from {CALIBRATION_FOLDER.name}...")
    
    for cal_file in cal_files:
        result = read_and_process_calibration_file(cal_file)
        if result:
            file_id = result['file_id']
            print(f"  ✓ {result['filename']:40} {file_id}: {result['baseline_currents']}")
            
            for ch_name, baseline_val in result['baseline_currents'].items():
                baseline_map[file_id][ch_name].append(baseline_val)
    
    # Average multiple files for same file_id
    averaged_baselines = {}
    for file_id, channels in baseline_map.items():
        averaged_baselines[file_id] = {}
        for ch_name, values in channels.items():
            avg_baseline = np.mean(values)
            averaged_baselines[file_id][ch_name] = float(avg_baseline)
            if len(values) > 1:
                print(f"  Averaged {len(values)} files for {file_id} {ch_name}: {avg_baseline:.6f} mA")
    
    return averaged_baselines


def read_and_calibrate_data_file(file_path, baseline_map):
    """Read a data file and apply baseline correction.
    
    Returns:
        dict: Calibrated data or None if error
    """
    filename = file_path.name
    file_id = extract_file_id(file_path)
    is_rfd11 = 'rfd11' in filename
    
    # Detect header row
    skip_rows = 4
    try:
        with open(file_path, 'r', errors='ignore') as f:
            for i in range(15):
                line = f.readline()
                if line.startswith('CH1'):
                    skip_rows = i
                    break
    except Exception:
        pass
    
    # Read channels
    try:
        if is_rfd11:
            df = pd.read_csv(file_path, skiprows=skip_rows, usecols=['CH1', 'CH2', 'CH3'])
            channels = ['CH1', 'CH2', 'CH3']
        else:
            df = pd.read_csv(file_path, skiprows=skip_rows, usecols=['CH1'])
            channels = ['CH1']
    except Exception as e:
        print(f"Error reading {filename}: {e}")
        return None
    
    # Get baseline for this file_id
    if file_id not in baseline_map:
        print(f"Warning: No baseline found for {file_id}, skipping {filename}")
        return None
    
    baselines = baseline_map[file_id]
    
    # Calibrate each channel
    calibrated_df = df.copy()
    
    for ch_name in channels:
        try:
            ch_data = df[ch_name].values.copy()
            
            # Convert raw data to current in mA
            current_mA = ch_data / 10 / 50 * 1000
            
            # Apply baseline correction
            baseline_mA = baselines.get(ch_name, 0.0)
            calibrated_current = current_mA - baseline_mA
            
            # Convert back to raw ADC counts
            calibrated_raw = calibrated_current * 10 * 50 / 1000
            calibrated_df[ch_name] = calibrated_raw
            
        except Exception as e:
            print(f"Error calibrating {filename} {ch_name}: {e}")
    
    return {
        'filename': filename,
        'file_id': file_id,
        'channels': channels,
        'calibrated_df': calibrated_df,
        'baselines_applied': baselines
    }


def apply_calibration_to_all_files(baseline_map):
    """Apply baseline calibration to all files in electrode_local folder."""
    
    if not DATA_FOLDER.exists():
        print(f"Error: Data folder not found: {DATA_FOLDER}")
        return
    
    OUTPUT_FOLDER.mkdir(exist_ok=True)
    
    data_files = sorted(DATA_FOLDER.glob("*.csv"))
    print(f"\nApplying calibration to {len(data_files)} files in {DATA_FOLDER.name}...")
    
    successful = 0
    failed = 0
    
    for data_file in data_files:
        result = read_and_calibrate_data_file(data_file, baseline_map)
        
        if result:
            # Save calibrated file
            output_file = OUTPUT_FOLDER / result['filename']
            try:
                result['calibrated_df'].to_csv(output_file, index=False)
                print(f"  ✓ {result['filename']:50} (baseline: {result['baselines_applied']})")
                successful += 1
            except Exception as e:
                print(f"  ✗ Error saving {result['filename']}: {e}")
                failed += 1
        else:
            failed += 1
    
    print(f"\nCalibration complete: {successful} successful, {failed} failed")
    print(f"Calibrated files saved to: {OUTPUT_FOLDER}")
    
    return successful, failed


def save_baseline_reference(baseline_map):
    """Save baseline reference for documentation."""
    try:
        with open(BASELINE_FILE, 'w') as f:
            json.dump(baseline_map, f, indent=2)
        print(f"\nBaseline reference saved to: {BASELINE_FILE}")
    except Exception as e:
        print(f"Error saving baseline reference: {e}")


# ===== MAIN EXECUTION =====
if __name__ == "__main__":
    print("="*70)
    print("ZERO CURRENT CALIBRATION")
    print("="*70)
    
    # Step 1: Calculate baseline from calibration files
    baseline_map = calculate_baseline_map()
    
    if not baseline_map:
        print("\nError: No baselines calculated. Exiting.")
        exit(1)
    
    print(f"\n{'='*70}")
    print(f"BASELINE SUMMARY")
    print(f"{'='*70}")
    for file_id in sorted(baseline_map.keys()):
        print(f"{file_id:20} {baseline_map[file_id]}")
    
    # Step 2: Apply calibration to all data files
    apply_calibration_to_all_files(baseline_map)
    
    # Step 3: Save baseline reference
    save_baseline_reference(baseline_map)
    
    print(f"\n{'='*70}")
    print("DONE")
    print(f"{'='*70}")


ZERO CURRENT CALIBRATION

Reading 16 calibration files from electrode_8mA...
  ✓ 20260131_220406_rfd11tbs1.csv            rfd11tbs1: {'CH1': 1.4792701817545428, 'CH2': 1.397807345183629, 'CH3': -0.9706322658066467}
  ✓ 20260131_220406_rfd11tbs2.csv            rfd11tbs2: {'CH1': -0.5779000475011877, 'CH2': -0.07577429435735895, 'CH3': 1.2053541338533469}
  ✓ 20260131_220518_rfd10tbs2.csv            rfd10tbs2: {'CH1': -0.9345946702664871}
  ✓ 20260131_220518_rfd10tbs3.csv            rfd10tbs3: {'CH1': 0.3358056097195141}
  ✓ 20260131_220518_rfd4tbs3.csv             rfd4tbs3: {'CH1': 0.8885257737113136}
  ✓ 20260131_220518_rfd4tbs4.csv             rfd4tbs4: {'CH1': -0.7674903754812269}
  ✓ 20260131_220518_rftb4tbs1.csv            rftb4tbs1: {'CH1': -0.054714064296785166}
  ✓ 20260131_220540_rfd10tbs1.csv            rfd10tbs1: {'CH1': 0.04710917772944323}
  ✓ 20260131_220540_rfd10tbs2.csv            rfd10tbs2: {'CH1': -0.9345946702664871}
  ✓ 20260131_220540_rfd10tbs3.csv            rfd10t